# SRE Incident Triage Agent - Interactive Walkthrough

Welcome to the interactive walkthrough for the **SRE Incident Triage Agent**.

This notebook demonstrates the collaborative execution of our 5 specialized sub-agents orchestrated by a central coordinator using the **Collaborative Teams** pattern.

### Agent Topology Overview
1. **Triage Agent**: Parses raw Sentry payloads, extracts metadata, and maps log levels to severities (`SEV1`-`SEV3`).
2. **Correlation Agent**: Queries runbook troubleshooting documentation via Google Developer Knowledge MCP and scans closed GitHub issues.
3. **RCA Agent**: Merges alert facts with the correlation context to draft a markdown Root Cause Analysis.
4. **Notifier Agent**: Formats the diagnostic results into a Slack notification template.
5. **HITL Gate Agent**: Enforces security guardrails. Evaluates settings and halts execution to prompt for operator approval (`y/n`) on any high-severity (`SEV1`/`SEV2`) remediation.

In [ ]:
# Configure path to allow imports from the root directory
import os
import sys
import json
sys.path.append(os.path.abspath('.'))

# Import core components
from capstone.mcp_client import get_sentry_event
from capstone.agents.triage_agent import classify_alert
from capstone.agents.correlation_agent import correlate
from capstone.agents.rca_agent import draft_rca
from capstone.agents.notifier_agent import format_slack_message
from capstone.agents.hitl_gate import evaluate

## Step 1: Simulating an Alert Event (Sentry Payload)
We fetch a simulated Sentry fatal OOM (Out Of Memory) event payload.

In [ ]:
event_id = "FAKE-EVENT-ID-001"
sentry_event = get_sentry_event(event_id)
print("Sentry Event Payload:")
print(json.dumps(sentry_event, indent=2))

## Step 2: The Triage Agent
The Triage Agent parses the raw payload, extracts metadata, identifies the target service (`payment-service`), and maps the `fatal` level to **SEV1** severity.

In [ ]:
triage_result = classify_alert(sentry_event)
print("Triage Output:")
print(json.dumps(triage_result, indent=2))

## Step 3: The Correlation Agent
Using the **Model Context Protocol (MCP)**, the Correlation Agent queries the `google-developer-knowledge` server for Google Cloud runbooks and searches the repository closed issues to find matching historical incidents.

In [ ]:
correlation_result = correlate(triage_result)
print("Correlation Output:")
print(json.dumps(correlation_result, indent=2))

## Step 4: The RCA Agent
The RCA Agent synthesizes the raw traceback, runbook context, and past incidents into a structured markdown Root Cause Analysis report.

In [ ]:
rca_result = draft_rca(correlation_result, triage_result)
print("Drafted Root Cause Analysis (RCA):\n")
print(rca_result["rca_draft"])

## Step 5: The Notifier Agent
The Notifier Agent formats the diagnostics and RCA summary into a Slack notification template for direct team consumption.

In [ ]:
notifier_result = format_slack_message(rca_result, triage_result)
print("Slack Notification Preview:\n")
print(notifier_result["slack_message"])

## Step 6: The Human-In-The-Loop (HITL) Gate Agent
For high-severity (SEV1/SEV2) actions, the HITL Gate Agent halts execution and prompts the console operator for approval (`y/n`) before any destructive command is run.

In [ ]:
proposed_action = f"kubectl rollout restart deployment/{triage_result['affected_service']}"

# Note: Running this cell will prompt you for approval in the input field below!
gate_result = evaluate(triage_result, proposed_action)
print("\nHITL Gate Output:")
print(json.dumps(gate_result, indent=2))

## Running the Full Orchestrated Pipeline
Alternatively, you can run the entire workflow end-to-end using the coordinator's orchestrate wrapper.

In [ ]:
from capstone.coordinator import orchestrate

# Run the full pipeline (this will also prompt the HITL gate)
incident_card = orchestrate(sentry_event)

## Running the Test Suite
The codebase features a robust pytest suite covering individual agent logic, MCP clients, telemetry, and pipeline integrations. You can run all 25 tests locally by executing:
```bash
python -m pytest -v
```